In [1]:
import torch
if not torch.cuda.is_available():
  raise Exception("GPU not available")
device = torch.device("cuda")
print(f"Device: {device}")

Device: cuda


In [2]:
import torch
import math
import time
import shutil
import psutil
import os

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    GPTQConfig,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

def get_process_memory_mb():
  """ Return how much RAM is the current program utilizing"""
  ## psutil --> Process and System Utilities
  process = psutil.Process(os.getpid()) ## os.getpid() --> Accessing the current process id!
  return process.memory_info().rss/1024 ** 2 ## rss --> Resident Set Size

def get_gpu_memory_mb():
  """Return current GPU memory in (MB) (if CUDA)"""
  if not torch.cuda.is_available():
    return 0.0
  return torch.cuda.memory_allocated() / 1024 ** 2

def describe_memory(label):
  cpu_mem = get_process_memory_mb()
  gpu_mem = get_gpu_memory_mb()
  print(f"[{label}] CPU memory: {cpu_mem:8.2f} MB | GPU memory: {gpu_mem:8.2f} MB")


@torch.no_grad()
def compute_perplexity(model, tokenizer,text: str) -> float:
  model.eval()
  enc = tokenizer(text, return_tensors="pt").to(device)
  outputs = model(**enc, labels=enc["input_ids"])
  return math.exp(outputs.loss.item())

@torch.no_grad()
def timed_generate(model,tokenizer,prompt:str,max_new_tokens: int = 40, num_runs: int = 3):
  model.eval()
  times = []
  last_output = True

  for i in range(num_runs):
    inputs = tokenizer(prompt,return_tensors = "pt").to(device)
    torch.cuda.empty_cache()
    start = time.perf_counter()
    out = model.generate(**inputs,max_new_tokens=max_new_tokens)
    end = time.perf_counter()
    times.append(end-start)
    last_output = tokenizer.decode(out[0], skip_special_tokens = True)
  avg_time = sum(times)/len(times)
  return avg_time,last_output

Device: cuda


In [3]:
model_id = "facebook/opt-125m"

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token_id is None:
  tokenizer.pad_token_id = tokenizer.eos_token_id

torch.cuda.empty_cache()
describe_memory("Before FP16 loading")
model_fp16 = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16,
).to(device)

describe_memory("After FP16 loading")
print("Baseline dtype:",next(model_fp16.parameters()).dtype)

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/441 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  251MB            

model.safetensors: downloading bytes:           |  0.00B            

[After FP16 loading] CPU memory:  1063.15 MB | GPU memory:   245.48 MB
Baseline dtype: torch.float16


In [6]:
calib_text = (
    "Quantization allows us to compress Large Language Models like GPT while"
    "Preserving most of their predictive power"
)
prompt = "In the future, Efficient Large Language Models will"
ppl_fp16 = compute_perplexity(model_fp16,tokenizer,calib_text)
t_fp16,out_fp16 = timed_generate(model_fp16,tokenizer,prompt)
print(f"FP16 Perplexity: {ppl_fp16:8.2f}")
print(f"Baseline fp16 gen time: {t_fp16:8.2f}")
print(f"Baseline fp16 output: {out_fp16}")

FP16 Perplexity:   562.09
Baseline fp16 gen time:     0.61
Baseline fp16 output: In the future, Efficient Large Language Models will be used to provide a more accurate representation of the size of the data.

The model will be used to provide a more accurate representation of the size of the data.

The model will


In [8]:
!pip install gptqmodel

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 52.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 4.5 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 215.0/215.0 kB 22.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.6/115.6 kB 13.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Us

In [7]:


# quant_config = {
#     "zero_point": True,
#     "q_group_size": 128,
#     "w_bit": 4,
#     "version": "GEMM",   # GEMM kernels: good general default
# }

torch.cuda.empty_cache()
describe_memory("Before AWQ quantization")

start = time.perf_counter()

awq_model = AutoModelForCausalLM.from_pretrained(
    "ybelkada/opt-125m-awq",
    low_cpu_mem_usage=True,
    use_cache=False,
)

# # Activation-aware quantization: uses a small calibration set internally
# awq_model.quantize(
#     tokenizer,
#     quant_config=quant_config,
# )

# Move quantized model to GPU for inference
awq_model.to(device)

end = time.perf_counter()
describe_memory("After AWQ quantization")


[Before AWQ quantization] CPU memory:  1599.50 MB | GPU memory:   255.23 MB


config.json:   0%|          | 0.00/979 [00:00<?, ?B/s]

ImportError: Loading an AWQ quantized model requires gptqmodel. Please install it with `pip install gptqmodel`

In [ ]:
ppl_awq = perplexity(awq_model, tokenizer, calib_text)
t_awq, out_awq = timed_generate(awq_model, tokenizer, prompt)

print(f"AWQ 4-bit perplexity: {ppl_awq:.3f}")
print(f"AWQ 4-bit gen time:   {t_awq:.3f} s")
print("AWQ 4-bit output:\n", out_awq)

print("\nSummary:")
print(f"  FP16 perplexity: {ppl_fp16:.3f}")
print(f"  AWQ  perplexity: {ppl_awq:.3f}")
print(f"  FP16 gen time:   {t_fp16:.3f} s")
print(f"  AWQ  gen time:   {t_awq:.3f} s")
